<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/PlantVillage_EfficientNetB2_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================
# 1. Kaggle Setup and Dataset Download
# ===============================
from google.colab import files
files.upload()  # Upload kaggle.json first

!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download PlantVillage dataset from Kaggle
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip -d PlantVillage

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
 93% 611M/658M [00:02<00:00, 203MB/s]
100% 658M/658M [00:06<00:00, 101MB/s]


In [ ]:
# ===============================
# 2. Imports
# ===============================
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras.preprocessing import image_dataset_from_directory

# Parameters
img_size = (260, 260)
batch_size = 32

# ===============================
# 3. Dataset Creation
# ===============================
# Train/Val/Test split (PlantVillage dataset comes only with raw folders, so we will split)
dataset = image_dataset_from_directory(
    "/content/PlantVillage/PlantVillage",
    image_size=img_size,
    batch_size=batch_size,
    validation_split=0.2,  # 80% train, 20% val
    subset="training",
    seed=123
)
val_dataset = image_dataset_from_directory(
    "/content/PlantVillage/PlantVillage",
    image_size=img_size,
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=123
)

# Split validation further into validation + test
val_batches = tf.data.experimental.cardinality(val_dataset)
test_dataset = val_dataset.take(val_batches // 2)
val_dataset = val_dataset.skip(val_batches // 2)

class_names = dataset.class_names
num_classes = len(class_names)
print("Classes:", class_names)
print("Num Classes:", num_classes)

# ===============================
# 4. Data Augmentation
# ===============================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

train_ds = dataset.map(lambda x, y: (data_augmentation(x), y))

# Prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_dataset = val_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

Found 20638 files belonging to 15 classes.
Using 16511 files for training.
Found 20638 files belonging to 15 classes.
Using 4127 files for validation.
Classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']
Num Classes: 15


In [ ]:
# ===============================
# 5. Model Building
# ===============================
base_model = EfficientNetB2(
    include_top=False,
    weights="imagenet",
    input_shape=(260, 260, 3)
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

def one_hot_map(images, labels):
    return images, tf.one_hot(labels, depth=num_classes)

train_ds_onehot = train_ds.map(one_hot_map)
val_ds_onehot = val_dataset.map(one_hot_map)
test_ds_onehot = test_dataset.map(one_hot_map)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [ ]:
# ===============================
# 6. Initial Training (Frozen Base)
# ===============================
history = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=10
)

NameError: name 'model' is not defined

In [ ]:
# ===============================
# 6.5 Random Search CV for Hyperparameter Tuning (Before Fine-Tuning)
# ===============================
from sklearn.model_selection import RandomizedSearchCV
from scikeras.wrappers import KerasClassifier
import numpy as np

def create_model(learning_rate=1e-3, dropout1=0.5, dropout2=0.4, dense_units=256, l2_reg=0.001):
    base_model = EfficientNetB2(
        include_top=False,
        weights="imagenet",
        input_shape=(260, 260, 3)
    )
    base_model.trainable = False  # Still frozen before fine-tuning

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(dropout1),
        layers.Dense(dense_units, activation="relu", kernel_regularizer=regularizers.l2(l2_reg)),
        layers.Dropout(dropout2),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"]
    )
    return model

# Wrap model for sklearn
keras_clf = KerasClassifier(model=create_model, epochs=5, batch_size=32, verbose=0)

# Hyperparameter search space
param_dist = {
    "model__learning_rate": [1e-2, 1e-3, 5e-4, 1e-4],
    "model__dropout1": [0.3, 0.4, 0.5, 0.6],
    "model__dropout2": [0.3, 0.4, 0.5],
    "model__dense_units": [128, 256, 512],
    "model__l2_reg": [1e-2, 1e-3, 1e-4]
}

# Prepare training data as numpy arrays (RandomizedSearchCV needs numpy, not tf.data)
X_train, y_train = [], []
for images, labels in train_ds_onehot.take(-1):  # collect all batches
    X_train.extend(images.numpy())
    y_train.extend(labels.numpy())
X_train, y_train = np.array(X_train), np.array(y_train)

# Run Random Search
random_search = RandomizedSearchCV(
    estimator=keras_clf,
    param_distributions=param_dist,
    n_iter=5,   # increase for more thorough search
    cv=3,
    verbose=2,
    n_jobs=1
)

random_search.fit(X_train, y_train)

print("Best Parameters from Random Search:", random_search.best_params_)
print("Best Cross-Validation Score:", random_search.best_score_)

In [ ]:
# ===============================
# 7. Fine-Tuning
# ===============================
base_model.trainable = True
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=20,
    initial_epoch=10
)

Epoch 11/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 456s 720ms/step - accuracy: 0.5395 - loss: 1.9724 - val_accuracy: 0.8384 - val_loss: 1.2468
Epoch 12/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 316s 612ms/step - accuracy: 0.7650 - loss: 1.4297 - val_accuracy: 0.8793 - val_loss: 1.1247
Epoch 13/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 315s 609ms/step - accuracy: 0.8219 - loss: 1.2768 - val_accuracy: 0.9192 - val_loss: 1.0588
Epoch 14/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 312s 604ms/step - accuracy: 0.8596 - loss: 1.2006 - val_accuracy: 0.9293 - val_loss: 1.0302
Epoch 15/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 310s 601ms/step - accuracy: 0.8867 - loss: 1.1478 - val_accuracy: 0.9375 - val_loss: 1.0040
Epoch 16/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 311s 602ms/step - accuracy: 0.8936 - loss: 1.1223 - val_accuracy: 0.9481 - val_loss: 0.9805
Epoch 17/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 316s 591ms/step - accuracy: 0.9144 - loss: 1.0808 - val_accuracy: 0.9505 - val_loss: 0.9686
Epoch 18/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 319s 618ms/step - accuracy: 

In [ ]:
# Save best model
best_model = random_search.best_estimator_.model_
best_model.save("PlantVillage_EfficientNetB2_Best_PreFineTune.h5")
print("Best pre-finetuning model saved as PlantVillage_EfficientNetB2_Best_PreFineTune.h5")

In [ ]:
# ===============================
# 8. Evaluation with Temperature Scaling
# ===============================
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize

TEMPERATURE = 2.0

def predict_with_temperature(model, dataset, T=1.0):
    all_probs, all_labels = [], []
    for images, labels in dataset:
        logits = model(images, training=False)
        scaled_logits = logits / T
        probs = tf.nn.softmax(scaled_logits).numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_probs)

y_true, y_probs = predict_with_temperature(model, test_ds_onehot, T=TEMPERATURE)
y_pred = np.argmax(y_probs, axis=1)

# Convert one-hot labels to integers
y_true_int = np.argmax(y_true, axis=1)

y_true_bin = label_binarize(y_true_int, classes=range(num_classes))

accuracy = accuracy_score(y_true_int, y_pred)
precision = precision_score(y_true_int, y_pred, average="weighted")
recall = recall_score(y_true_int, y_pred, average="weighted")
f1 = f1_score(y_true_int, y_pred, average="weighted")
kappa = cohen_kappa_score(y_true_int, y_pred)
auc = roc_auc_score(y_true_bin, y_probs, average="macro", multi_class="ovr")

print("\n Evaluation Metrics After Calibration:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Kappa    : {kappa:.4f}")


 Evaluation Metrics After Calibration:
Accuracy : 0.9731
Precision: 0.9739
Recall   : 0.9731
F1 Score : 0.9730
AUC      : 0.9994
Kappa    : 0.9706
